In [1]:
# ---------------- 0. Imports & Config ----------------

# Core Python utilities
import json
import io
import os
import random
import time
from pathlib import Path
from collections import defaultdict

# Data processing
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# ML / retrieval
import torch
import faiss

# Google Cloud
from google.cloud import storage

# Transformers: same as 04_retrieval_experiments_clean.ipynb
from transformers import (
    CLIPProcessor,
    CLIPModel,
    BlipProcessor,
    BlipForImageTextRetrieval,
)

# ---------------- GCS config ----------------
BUCKET_NAME = "egorecall-data"

# ---------------- Local paths ----------------
WORK_DIR = Path("/home/jupyter/egorecall_retrieval_subset")
RESULTS_DIR = Path("/home/jupyter/results/retrieval_subset")
TMP_DIR = Path("/home/jupyter/tmp/retrieval_work")
FRAME_DIR = TMP_DIR / "frames"

for d in [WORK_DIR, RESULTS_DIR, TMP_DIR, FRAME_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------- N_CLIPS=300 preprocessing config ----------------
N_CLIPS = 300
RANDOM_SEED = 42

BATCH_SIZE_CLIPS = 60
BATCH_ID = 0  # Change this later: 0, 1, 2, 3, 4

# ---------------- Model config: same as 04 baseline ----------------
CLIP_MODEL = "openai/clip-vit-base-patch32"
BLIP_MODEL = "Salesforce/blip-itm-base-coco"
BATCH_SIZE = 64

# ---------------- Frame extraction config ----------------
VIDEO_FPS = 30
INDEX_STRIDE = 30  # 1 FPS if source video is 30 FPS
TOLERANCE_SEC = 5
TOLERANCE_FRAMES = TOLERANCE_SEC * VIDEO_FPS

# ---------------- Video cleanup policy ----------------
# False means: after each batch is fully processed, we can remove the videos used by that batch.
# This keeps disk usage under control.
KEEP_VIDEOS_AFTER_BATCH = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

print("N_CLIPS:", N_CLIPS)
print("BATCH_ID:", BATCH_ID)
print("BATCH_SIZE_CLIPS:", BATCH_SIZE_CLIPS)
print("Work dir:", WORK_DIR)
print("Temp dir:", TMP_DIR)
print("Frame dir:", FRAME_DIR)
print("Results dir:", RESULTS_DIR)

Device : cuda
GPU    : Tesla T4
N_CLIPS: 300
BATCH_ID: 0
BATCH_SIZE_CLIPS: 60
Work dir: /home/jupyter/egorecall_retrieval_subset
Temp dir: /home/jupyter/tmp/retrieval_work
Frame dir: /home/jupyter/tmp/retrieval_work/frames
Results dir: /home/jupyter/results/retrieval_subset


In [2]:
# ---------------- 0. Imports & Config ----------------

# Core Python utilities
import json
import io
import os
import random
import time
from pathlib import Path
from collections import defaultdict

# Data processing
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

# ML / retrieval
import torch
import faiss

# Google Cloud
from google.cloud import storage

# Transformers: same as 04_retrieval_experiments_clean.ipynb
from transformers import (
    CLIPProcessor,
    CLIPModel,
    BlipProcessor,
    BlipForImageTextRetrieval,
)

# ---------------- GCS config ----------------
BUCKET_NAME = "egorecall-data"

# ---------------- Local paths ----------------
WORK_DIR = Path("/home/jupyter/egorecall_retrieval_subset")
RESULTS_DIR = Path("/home/jupyter/results/retrieval_subset")
TMP_DIR = Path("/home/jupyter/tmp/retrieval_work")
FRAME_DIR = TMP_DIR / "frames"

for d in [WORK_DIR, RESULTS_DIR, TMP_DIR, FRAME_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------- N_CLIPS=300 preprocessing config ----------------
N_CLIPS = 300
RANDOM_SEED = 42

BATCH_SIZE_CLIPS = 60
BATCH_ID = 0  # Change this later: 0, 1, 2, 3, 4

# ---------------- Model config: same as 04 baseline ----------------
CLIP_MODEL = "openai/clip-vit-base-patch32"
BLIP_MODEL = "Salesforce/blip-itm-base-coco"
BATCH_SIZE = 64

# ---------------- Frame extraction config ----------------
VIDEO_FPS = 30
INDEX_STRIDE = 30  # 1 FPS if source video is 30 FPS
TOLERANCE_SEC = 5
TOLERANCE_FRAMES = TOLERANCE_SEC * VIDEO_FPS

# ---------------- Video cleanup policy ----------------
# False means: after each batch is fully processed, we can remove the videos used by that batch.
# This keeps disk usage under control.
KEEP_VIDEOS_AFTER_BATCH = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

print("N_CLIPS:", N_CLIPS)
print("BATCH_ID:", BATCH_ID)
print("BATCH_SIZE_CLIPS:", BATCH_SIZE_CLIPS)
print("Work dir:", WORK_DIR)
print("Temp dir:", TMP_DIR)
print("Frame dir:", FRAME_DIR)
print("Results dir:", RESULTS_DIR)

Device : cuda
GPU    : Tesla T4
N_CLIPS: 300
BATCH_ID: 0
BATCH_SIZE_CLIPS: 60
Work dir: /home/jupyter/egorecall_retrieval_subset
Temp dir: /home/jupyter/tmp/retrieval_work
Frame dir: /home/jupyter/tmp/retrieval_work/frames
Results dir: /home/jupyter/results/retrieval_subset


In [3]:
# ---------------- 1. Connect to GCS and Load Metadata ----------------

gcs_client = storage.Client()
bucket = gcs_client.bucket(BUCKET_NAME)

print(f"Connected to gs://{BUCKET_NAME}")

metadata_path = WORK_DIR / "vq_query_sets.parquet"
annotation_path = WORK_DIR / "vq_val.json"

if not metadata_path.exists():
    print("Downloading processed/vq_query_sets.parquet...")
    bucket.blob("processed/vq_query_sets.parquet").download_to_filename(str(metadata_path))
else:
    print("Using local metadata:", metadata_path)

if not annotation_path.exists():
    print("Downloading ego4d/v2/annotations/vq_val.json...")
    bucket.blob("ego4d/v2/annotations/vq_val.json").download_to_filename(str(annotation_path))
else:
    print("Using local annotation:", annotation_path)

df = pd.read_parquet(metadata_path)

with open(annotation_path, "r") as f:
    val_raw = json.load(f)

val_df = df[df["split"] == "val"].copy()

print("All query-sets:", len(df))
print("Validation query-sets:", len(val_df))
print("Validation clips:", val_df["clip_uid"].nunique())
print("Columns:", df.columns.tolist())

display(
    val_df[
        [
            "split",
            "clip_uid",
            "video_uid",
            "annotation_uid",
            "qs_id",
            "object_title",
            "query_video_frame",
            "response_track_len",
            "vc_area_norm",
        ]
    ].head(5)
)

Connected to gs://egorecall-data
Using local metadata: /home/jupyter/egorecall_retrieval_subset/vq_query_sets.parquet
Using local annotation: /home/jupyter/egorecall_retrieval_subset/vq_val.json
All query-sets: 18114
Validation query-sets: 4507
Validation clips: 1157
Columns: ['split', 'video_uid', 'annotation_uid', 'qs_id', 'object_title', 'query_frame', 'query_video_frame', 'is_valid', 'has_errors', 'has_warnings', 'response_track_len', 'vc_frame', 'vc_x', 'vc_y', 'vc_w', 'vc_h', 'orig_w', 'orig_h', 'clip_uid', 'source_clip_uid', 'clip_fps', 'clip_start_sec', 'clip_end_sec', 'clip_duration_sec', 'video_start_sec', 'video_end_sec', 'annotation_complete', 'vc_area_norm', 'vc_cx_norm', 'vc_cy_norm', 'vc_aspect', 'clip_total_frames', 'query_pos_norm']


,split,clip_uid,video_uid,annotation_uid,qs_id,object_title,query_video_frame,response_track_len,vc_area_norm
13607,val,d5935c29-1b8d-417d-9bbb-5ebd47e9256d,e14e03f8-13e4-4df2-87b0-e1ad8a175f7c,3760b439-d94b-4d76-8b39-1bb355800649,3,sellotape,8598.0,59,0.015786
13608,val,d5935c29-1b8d-417d-9bbb-5ebd47e9256d,e14e03f8-13e4-4df2-87b0-e1ad8a175f7c,3760b439-d94b-4d76-8b39-1bb355800649,1,container,480.0,26,0.004053
13609,val,d5935c29-1b8d-417d-9bbb-5ebd47e9256d,e14e03f8-13e4-4df2-87b0-e1ad8a175f7c,3760b439-d94b-4d76-8b39-1bb355800649,2,cotton bud,10722.0,30,0.035157
13610,val,dfe962ab-6aa7-4888-9378-796817a30ab6,e14e03f8-13e4-4df2-87b0-e1ad8a175f7c,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,42288.0,30,0.066873
13611,val,dfe962ab-6aa7-4888-9378-796817a30ab6,e14e03f8-13e4-4df2-87b0-e1ad8a175f7c,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,46764.0,7,0.005119


In [4]:
# ---------------- 2. Sample N_CLIPS=300 validation clips ----------------

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Use string clip_uids to avoid float/string sorting issues.
all_val_clips = val_df["clip_uid"].dropna().astype(str).unique().tolist()

subset_clips = random.sample(
    all_val_clips,
    min(N_CLIPS, len(all_val_clips))
)

subset_clip_set = set(subset_clips)
subset_df = val_df[val_df["clip_uid"].astype(str).isin(subset_clip_set)].copy()

print("Sampled clips:", len(subset_clips))
print("Query-sets in subset:", len(subset_df))

display(
    subset_df.groupby("clip_uid")
    .size()
    .reset_index(name="num_query_sets")
    .sort_values("num_query_sets", ascending=False)
    .head(10)
)

Sampled clips: 300
Query-sets in subset: 1157


,clip_uid,num_query_sets
189,ad9083b3-9b0b-4c27-a941-38f57be13edf,12
15,1004f7fe-c397-4123-bad9-e02df2e154dd,6
281,f14ea4c5-2a1a-4855-a212-3b2abe7aa0f2,6
286,f54175da-19e5-4ee7-97b3-75baa247b713,6
287,f552173d-229e-496d-b414-3708df4c842e,6
280,f1350714-1aa5-421b-a268-690110ca9c06,6
10,09d85b55-d473-49c1-9ebc-e871714785cd,6
208,b8981cdd-a897-41b6-b1ef-e9957d270e9e,6
209,b8e1c869-b304-49fc-ba5f-9a5ae3b3770c,6
293,f9cbfb3a-d2e2-4f30-9c01-ee341b5e8887,6


In [5]:
# ---------------- Build workplan: clip_uid -> video info + query-sets ----------------

workplan = {}
rt_lookup = {}  # (annotation_uid, qs_id) -> [video_frame_numbers]

for video in val_raw["videos"]:
    video_uid = video["video_uid"]

    for clip in video["clips"]:
        clip_uid = str(clip["clip_uid"])

        if clip_uid not in subset_clip_set:
            continue

        workplan[clip_uid] = {
            "video_uid": video_uid,
            "video_start_frame": clip["video_start_frame"],
            "video_end_frame": clip["video_end_frame"],
            "query_sets": [],
        }

        for anno in clip["annotations"]:
            annotation_uid = anno["annotation_uid"]

            for qs_id, qs in anno["query_sets"].items():
                if not qs.get("is_valid"):
                    continue

                workplan[clip_uid]["query_sets"].append({
                    "annotation_uid": annotation_uid,
                    "qs_id": str(qs_id),
                    "vc_blob": (
                        f"frames/retrieval/visual_crops/val/{clip_uid}/"
                        f"{annotation_uid}_{qs_id}.jpg"
                    ),
                    "object_title": qs.get("object_title", "").strip().lower(),
                })

                key = (annotation_uid, str(qs_id))
                rt_lookup[key] = [
                    b["video_frame_number"]
                    for b in qs.get("response_track", [])
                    if "video_frame_number" in b
                ]

print("Workplan clips:", len(workplan))
print("Response-track entries:", len(rt_lookup))

example_clip_uid = next(iter(workplan))
example_entry = workplan[example_clip_uid]

print("\nExample clip_uid:", example_clip_uid)
print("Example video_uid:", example_entry["video_uid"])
print("Video start frame:", example_entry["video_start_frame"])
print("Video end frame:", example_entry["video_end_frame"])
print("Number of query-sets:", len(example_entry["query_sets"]))
print("\nFirst query-set:")
print(example_entry["query_sets"][0])

Workplan clips: 300
Response-track entries: 1157

Example clip_uid: dfe962ab-6aa7-4888-9378-796817a30ab6
Example video_uid: e14e03f8-13e4-4df2-87b0-e1ad8a175f7c
Video start frame: 40500
Video end frame: 49500
Number of query-sets: 6

First query-set:
{'annotation_uid': '3b057788-ad90-432d-90ff-de1c45361df6', 'qs_id': '2', 'vc_blob': 'frames/retrieval/visual_crops/val/dfe962ab-6aa7-4888-9378-796817a30ab6/3b057788-ad90-432d-90ff-de1c45361df6_2.jpg', 'object_title': 'portable fan'}


In [6]:
# ---------------- 3. Cache directories ----------------
# Same saving targets as the 20-clip run.

CACHE_DIR = RESULTS_DIR / "cache"
INDEX_CACHE_DIR = CACHE_DIR / "index_frame_embeddings"
QUERY_CACHE_DIR = CACHE_DIR / "query_embeddings"

for d in [CACHE_DIR, INDEX_CACHE_DIR, QUERY_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Cache dir:", CACHE_DIR)
print("Index embedding cache dir:", INDEX_CACHE_DIR)
print("Query embedding cache dir:", QUERY_CACHE_DIR)

existing_index_caches = list(INDEX_CACHE_DIR.glob("*_index_embeddings.npz"))
existing_query_caches = list(QUERY_CACHE_DIR.glob("*_query_embeddings.npz"))
existing_videos = list(TMP_DIR.glob("*.mp4"))

print("Existing index caches:", len(existing_index_caches))
print("Existing query caches:", len(existing_query_caches))
print("Existing videos:", len(existing_videos))

Cache dir: /home/jupyter/results/retrieval_subset/cache
Index embedding cache dir: /home/jupyter/results/retrieval_subset/cache/index_frame_embeddings
Query embedding cache dir: /home/jupyter/results/retrieval_subset/cache/query_embeddings
Existing index caches: 299
Existing query caches: 299
Existing videos: 48


In [7]:
# ---------------- Select one batch from the 300-clip workplan ----------------

all_workplan_clips = list(workplan.keys())

start_idx = BATCH_ID * BATCH_SIZE_CLIPS
end_idx = min(start_idx + BATCH_SIZE_CLIPS, len(all_workplan_clips))

batch_clip_uids = all_workplan_clips[start_idx:end_idx]

print("Total workplan clips:", len(all_workplan_clips))
print("Batch ID:", BATCH_ID)
print("Batch start:", start_idx)
print("Batch end:", end_idx)
print("Batch clips:", len(batch_clip_uids))
print("First 5 batch clips:", batch_clip_uids[:5])

Total workplan clips: 300
Batch ID: 0
Batch start: 0
Batch end: 60
Batch clips: 60
First 5 batch clips: ['dfe962ab-6aa7-4888-9378-796817a30ab6', 'b157270d-c67a-41db-91ec-32bb3ec738fc', '7bb41f34-1743-42ad-97d1-c54f4472b85e', 'ea365183-5841-4f00-9a2e-1b81c5e09c98', 'b961eb69-cfdd-4d5c-b427-4b667fedfcda']


In [8]:
# ---------------- 4. Load CLIP and BLIP models ----------------
# Same as 04_retrieval_experiments_clean.ipynb.

print("Loading CLIP...")
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL).to(DEVICE)
clip_model.eval()

print("Loading BLIP...")
blip_processor = BlipProcessor.from_pretrained(BLIP_MODEL)
blip_model = BlipForImageTextRetrieval.from_pretrained(BLIP_MODEL).to(DEVICE)
blip_model.eval()

print("Models loaded.")
print("clip_model type:", type(clip_model))
print("blip_model type:", type(blip_model))

Loading CLIP...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading BLIP...


Loading weights:   0%|          | 0/472 [00:00<?, ?it/s]

Models loaded.
clip_model type: <class 'transformers.models.clip.modeling_clip.CLIPModel'>
blip_model type: <class 'transformers.models.blip.modeling_blip.BlipForImageTextRetrieval'>


In [9]:
# ---------------- 5. Helper functions from 04 pipeline ----------------

def download_video_if_needed(video_uid, video_dir=TMP_DIR):
    """
    Download the source video from GCS if it does not already exist locally.
    Returns the local video path.
    """
    video_dir.mkdir(parents=True, exist_ok=True)
    local_video_path = video_dir / f"{video_uid}.mp4"

    if local_video_path.exists():
        print(f"Video already exists locally: {local_video_path}")
        return local_video_path

    video_blob = f"ego4d/v2/video_540ss/{video_uid}.mp4"
    print(f"Downloading video from gs://{BUCKET_NAME}/{video_blob}")

    bucket.blob(video_blob).download_to_filename(str(local_video_path))

    print(f"Downloaded video to: {local_video_path}")
    print(f"Video size: {local_video_path.stat().st_size / (1024 ** 2):.2f} MB")

    return local_video_path


def extract_index_frames(video_path, clip_uid, start_frame, end_frame, stride=INDEX_STRIDE):
    """
    Extract index frames from a source video for one clip.
    Frames are saved as {frame_number}.jpg.
    Returns a DataFrame with extracted frame paths.
    """
    output_dir = FRAME_DIR / clip_uid
    output_dir.mkdir(parents=True, exist_ok=True)

    target_frames = list(range(int(start_frame), int(end_frame), int(stride)))

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {video_path}")

    extracted_rows = []

    for frame_number in tqdm(target_frames, desc=f"Extracting frames for {clip_uid}", leave=False):
        output_path = output_dir / f"{frame_number:06d}.jpg"

        if output_path.exists():
            extracted_rows.append({
                "clip_uid": clip_uid,
                "frame_number": frame_number,
                "frame_path": str(output_path),
                "status": "exists",
            })
            continue

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        success, frame_bgr = cap.read()

        if not success:
            extracted_rows.append({
                "clip_uid": clip_uid,
                "frame_number": frame_number,
                "frame_path": None,
                "status": "failed",
            })
            continue

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        Image.fromarray(frame_rgb).save(output_path, quality=90)

        extracted_rows.append({
            "clip_uid": clip_uid,
            "frame_number": frame_number,
            "frame_path": str(output_path),
            "status": "extracted",
        })

    cap.release()

    return pd.DataFrame(extracted_rows)


def ensure_index_frames_for_clip(clip_uid):
    """
    Ensure index frames exist for one clip.
    If frames already exist, extract_index_frames will skip them.
    """
    clip_info = workplan[clip_uid]
    video_uid = clip_info["video_uid"]

    video_path = download_video_if_needed(video_uid)

    extraction_df = extract_index_frames(
        video_path=video_path,
        clip_uid=clip_uid,
        start_frame=clip_info["video_start_frame"],
        end_frame=clip_info["video_end_frame"],
        stride=INDEX_STRIDE,
    )

    available = extraction_df[extraction_df["status"].isin(["extracted", "exists"])]

    if len(available) == 0:
        raise RuntimeError(f"No index frames available for clip: {clip_uid}")

    return extraction_df


def load_index_frames_for_clip(clip_uid):
    """
    Load extracted index frame images for a clip.

    Returns:
        frame_numbers: np.ndarray
        images: list of PIL images
        frame_paths: list of local image paths
    """
    clip_frame_dir = FRAME_DIR / clip_uid

    if not clip_frame_dir.exists() or len(list(clip_frame_dir.glob("*.jpg"))) == 0:
        ensure_index_frames_for_clip(clip_uid)

    frame_paths = sorted(clip_frame_dir.glob("*.jpg"))

    if len(frame_paths) == 0:
        raise FileNotFoundError(f"No extracted frame images found in: {clip_frame_dir}")

    frame_numbers = []
    images = []

    for path in frame_paths:
        frame_number = int(path.stem)
        img = Image.open(path).convert("RGB")

        frame_numbers.append(frame_number)
        images.append(img)

    frame_numbers = np.array(frame_numbers, dtype=np.int32)

    return frame_numbers, images, frame_paths


def _to_feature_tensor(output):
    """
    Convert Hugging Face model output into a tensor embedding.
    Same robust logic as 04 notebook.
    """
    if isinstance(output, torch.Tensor):
        return output

    if hasattr(output, "image_embeds") and output.image_embeds is not None:
        return output.image_embeds

    if hasattr(output, "pooler_output") and output.pooler_output is not None:
        return output.pooler_output

    if hasattr(output, "last_hidden_state") and output.last_hidden_state is not None:
        return output.last_hidden_state[:, 0, :]

    raise ValueError(f"Cannot extract tensor from output type: {type(output)}")


def embed_images(images, batch_size=BATCH_SIZE):
    """
    Embed a list of PIL images with both CLIP and BLIP.

    Returns:
        clip_embs: np.ndarray of shape (N, D_clip)
        blip_embs: np.ndarray of shape (N, D_blip)
    """
    clip_embs = []
    blip_embs = []

    for i in tqdm(range(0, len(images), batch_size), desc="Embedding images", leave=False):
        batch = images[i:i + batch_size]

        with torch.no_grad():
            # ----- CLIP -----
            clip_inputs = clip_processor(
                images=batch,
                return_tensors="pt",
                padding=True,
            ).to(DEVICE)

            clip_out = clip_model.get_image_features(**clip_inputs)
            clip_feat = _to_feature_tensor(clip_out)
            clip_feat = clip_feat / clip_feat.norm(dim=-1, keepdim=True)
            clip_embs.append(clip_feat.detach().cpu().numpy())

            # ----- BLIP -----
            blip_inputs = blip_processor(
                images=batch,
                return_tensors="pt",
                padding=True,
            ).to(DEVICE)

            blip_out = blip_model.vision_model(pixel_values=blip_inputs["pixel_values"])
            blip_feat = _to_feature_tensor(blip_out)
            blip_feat = blip_feat / blip_feat.norm(dim=-1, keepdim=True)
            blip_embs.append(blip_feat.detach().cpu().numpy())

    clip_embs = np.vstack(clip_embs).astype(np.float32)
    blip_embs = np.vstack(blip_embs).astype(np.float32)

    return clip_embs, blip_embs


def load_visual_crops_for_clip(clip_uid):
    """
    Load visual crop query images for one clip from GCS.

    Returns:
        query_images: list of PIL images
        query_metadata: list of dicts
    """
    clip_info = workplan[clip_uid]

    query_images = []
    query_metadata = []

    for qs in clip_info["query_sets"]:
        vc_blob = qs["vc_blob"]

        try:
            blob_data = bucket.blob(vc_blob).download_as_bytes()
            img = Image.open(io.BytesIO(blob_data)).convert("RGB")

            query_images.append(img)
            query_metadata.append({
                "clip_uid": clip_uid,
                "annotation_uid": qs["annotation_uid"],
                "qs_id": str(qs["qs_id"]),
                "object_title": qs["object_title"],
                "vc_blob": vc_blob,
            })

        except Exception as e:
            print(f"Failed to load visual crop: {vc_blob}")
            print("Error:", e)

    return query_images, query_metadata


def build_or_load_index_embedding_cache(clip_uid, force_recompute=False):
    """
    Build or load cached CLIP/BLIP embeddings for index frames of one clip.
    """
    cache_path = INDEX_CACHE_DIR / f"{clip_uid}_index_embeddings.npz"

    if cache_path.exists() and not force_recompute:
        print(f"Loading cached index embeddings: {cache_path}")
        data = np.load(cache_path)
        return {
            "frame_numbers": data["frame_numbers"],
            "clip_embs": data["clip_embs"],
            "blip_embs": data["blip_embs"],
        }

    print(f"Building index embedding cache for clip: {clip_uid}")

    frame_numbers, images, frame_paths = load_index_frames_for_clip(clip_uid)

    print("Number of index frames:", len(images))

    clip_embs, blip_embs = embed_images(images)

    np.savez_compressed(
        cache_path,
        frame_numbers=frame_numbers,
        clip_embs=clip_embs,
        blip_embs=blip_embs,
    )

    print(f"Saved index embeddings to: {cache_path}")
    print("CLIP index embedding shape:", clip_embs.shape)
    print("BLIP index embedding shape:", blip_embs.shape)

    return {
        "frame_numbers": frame_numbers,
        "clip_embs": clip_embs,
        "blip_embs": blip_embs,
    }


def build_or_load_query_embedding_cache(clip_uid, force_recompute=False):
    """
    Build or load cached CLIP/BLIP embeddings for visual crop query images.
    """
    cache_path = QUERY_CACHE_DIR / f"{clip_uid}_query_embeddings.npz"
    metadata_path = QUERY_CACHE_DIR / f"{clip_uid}_query_metadata.parquet"

    if cache_path.exists() and metadata_path.exists() and not force_recompute:
        print(f"Loading cached query embeddings: {cache_path}")
        data = np.load(cache_path)
        query_metadata_df = pd.read_parquet(metadata_path)

        return {
            "query_metadata": query_metadata_df,
            "clip_embs": data["clip_embs"],
            "blip_embs": data["blip_embs"],
        }

    print(f"Building query embedding cache for clip: {clip_uid}")

    query_images, query_metadata = load_visual_crops_for_clip(clip_uid)

    if len(query_images) == 0:
        raise ValueError(f"No visual crop query images loaded for clip: {clip_uid}")

    print("Number of visual crop queries:", len(query_images))

    clip_embs, blip_embs = embed_images(query_images)

    query_metadata_df = pd.DataFrame(query_metadata)

    np.savez_compressed(
        cache_path,
        clip_embs=clip_embs,
        blip_embs=blip_embs,
    )

    query_metadata_df.to_parquet(metadata_path, index=False)

    print(f"Saved query embeddings to: {cache_path}")
    print(f"Saved query metadata to: {metadata_path}")
    print("CLIP query embedding shape:", clip_embs.shape)
    print("BLIP query embedding shape:", blip_embs.shape)

    return {
        "query_metadata": query_metadata_df,
        "clip_embs": clip_embs,
        "blip_embs": blip_embs,
    }


print("Helper functions defined.")

Helper functions defined.


In [14]:
# ---------------- 6. Quick test on one clip ----------------

test_clip_uid = batch_clip_uids[0]

print("Test clip:", test_clip_uid)
print("Test video:", workplan[test_clip_uid]["video_uid"])
print("Query count:", len(workplan[test_clip_uid]["query_sets"]))

# 1. Ensure/load index frames
idx_frame_numbers, idx_images, idx_frame_paths = load_index_frames_for_clip(test_clip_uid)

print("Loaded index frames:", len(idx_images))
print("First frame number:", idx_frame_numbers[0])
print("Last frame number:", idx_frame_numbers[-1])
print("First image size:", idx_images[0].size)

# 2. Test embedding on only 2 images
test_clip_embs, test_blip_embs = embed_images(idx_images[:2])

print("CLIP embedding shape:", test_clip_embs.shape)
print("BLIP embedding shape:", test_blip_embs.shape)
print("CLIP first norm:", np.linalg.norm(test_clip_embs[0]))
print("BLIP first norm:", np.linalg.norm(test_blip_embs[0]))

# 3. Test query crop loading
query_images, query_metadata = load_visual_crops_for_clip(test_clip_uid)

print("Loaded query images:", len(query_images))
display(pd.DataFrame(query_metadata).head())

# 4. Test query embedding on up to 2 query images
q_clip_embs, q_blip_embs = embed_images(query_images[:2])

print("Query CLIP embedding shape:", q_clip_embs.shape)
print("Query BLIP embedding shape:", q_blip_embs.shape)
print("Query CLIP first norm:", np.linalg.norm(q_clip_embs[0]))
print("Query BLIP first norm:", np.linalg.norm(q_blip_embs[0]))

Test clip: dfe962ab-6aa7-4888-9378-796817a30ab6
Test video: e14e03f8-13e4-4df2-87b0-e1ad8a175f7c
Query count: 6
Loaded index frames: 300
First frame number: 40500
Last frame number: 49470
First image size: (960, 540)


Embedding images:   0%|          | 0/1 [00:00<?, ?it/s]

CLIP embedding shape: (2, 512)
BLIP embedding shape: (2, 768)
CLIP first norm: 1.0
BLIP first norm: 1.0
Loaded query images: 6


,clip_uid,annotation_uid,qs_id,object_title,vc_blob
0,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,2,portable fan,frames/retrieval/visual_crops/val/dfe962ab-6aa...
1,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,3,tube,frames/retrieval/visual_crops/val/dfe962ab-6aa...
2,dfe962ab-6aa7-4888-9378-796817a30ab6,3b057788-ad90-432d-90ff-de1c45361df6,1,green bowl,frames/retrieval/visual_crops/val/dfe962ab-6aa...
3,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,2,stool,frames/retrieval/visual_crops/val/dfe962ab-6aa...
4,dfe962ab-6aa7-4888-9378-796817a30ab6,dc29a4cf-1430-43fe-98d2-156ca2b303cf,3,bucket,frames/retrieval/visual_crops/val/dfe962ab-6aa...


Embedding images:   0%|          | 0/1 [00:00<?, ?it/s]

Query CLIP embedding shape: (2, 512)
Query BLIP embedding shape: (2, 768)
Query CLIP first norm: 1.0
Query BLIP first norm: 0.99999994


In [12]:
# ---------------- 7. Cache status helper ----------------

def get_index_cache_path(clip_uid):
    return INDEX_CACHE_DIR / f"{clip_uid}_index_embeddings.npz"

def get_query_cache_path(clip_uid):
    return QUERY_CACHE_DIR / f"{clip_uid}_query_embeddings.npz"

def get_query_metadata_path(clip_uid):
    return QUERY_CACHE_DIR / f"{clip_uid}_query_metadata.parquet"

def check_clip_cache_status(clip_uid):
    index_cache_path = get_index_cache_path(clip_uid)
    query_cache_path = get_query_cache_path(clip_uid)
    query_metadata_path = get_query_metadata_path(clip_uid)

    return {
        "clip_uid": clip_uid,
        "index_cache_exists": index_cache_path.exists(),
        "query_cache_exists": query_cache_path.exists(),
        "query_metadata_exists": query_metadata_path.exists(),
        "fully_cached": (
            index_cache_path.exists()
            and query_cache_path.exists()
            and query_metadata_path.exists()
        ),
    }

batch_cache_status_df = pd.DataFrame(
    [check_clip_cache_status(clip_uid) for clip_uid in batch_clip_uids]
)

display(batch_cache_status_df.head())

print("Batch clips:", len(batch_cache_status_df))
print("Index caches exist:", batch_cache_status_df["index_cache_exists"].sum())
print("Query caches exist:", batch_cache_status_df["query_cache_exists"].sum())
print("Query metadata exists:", batch_cache_status_df["query_metadata_exists"].sum())
print("Fully cached:", batch_cache_status_df["fully_cached"].sum())

,clip_uid,index_cache_exists,query_cache_exists,query_metadata_exists,fully_cached
0,dfe962ab-6aa7-4888-9378-796817a30ab6,True,True,True,True
1,b157270d-c67a-41db-91ec-32bb3ec738fc,True,True,True,True
2,7bb41f34-1743-42ad-97d1-c54f4472b85e,True,True,True,True
3,ea365183-5841-4f00-9a2e-1b81c5e09c98,True,True,True,True
4,b961eb69-cfdd-4d5c-b427-4b667fedfcda,True,True,True,True


Batch clips: 60
Index caches exist: 60
Query caches exist: 60
Query metadata exists: 60
Fully cached: 60


In [13]:
# ---------------- 8. Preprocess one clip safely ----------------

def preprocess_one_clip_for_cache(clip_uid, force_recompute=False):
    """
    Build the same cache objects as the 20-clip run:
    1. index frame embeddings npz
    2. query visual crop embeddings npz
    3. query metadata parquet

    If all cache files already exist, skip this clip.
    """
    status_before = check_clip_cache_status(clip_uid)

    row = {
        "clip_uid": clip_uid,
        "video_uid": workplan[clip_uid]["video_uid"],
        "status": "started",
        "error": None,
        "index_cache_exists_before": status_before["index_cache_exists"],
        "query_cache_exists_before": status_before["query_cache_exists"],
        "query_metadata_exists_before": status_before["query_metadata_exists"],
    }

    try:
        if status_before["fully_cached"] and not force_recompute:
            row["status"] = "skipped_fully_cached"
            row["num_index_frames"] = None
            row["num_queries"] = len(workplan[clip_uid]["query_sets"])
            return row

        index_cache = build_or_load_index_embedding_cache(
            clip_uid,
            force_recompute=force_recompute,
        )

        query_cache = build_or_load_query_embedding_cache(
            clip_uid,
            force_recompute=force_recompute,
        )

        status_after = check_clip_cache_status(clip_uid)

        row.update({
            "status": "success",
            "num_index_frames": len(index_cache["frame_numbers"]),
            "num_queries": len(query_cache["query_metadata"]),
            "index_cache_exists_after": status_after["index_cache_exists"],
            "query_cache_exists_after": status_after["query_cache_exists"],
            "query_metadata_exists_after": status_after["query_metadata_exists"],
            "fully_cached_after": status_after["fully_cached"],
        })

        return row

    except Exception as e:
        row["status"] = "error"
        row["error"] = repr(e)
        return row

In [15]:
# ---------------- Test one clip only ----------------

test_clip_uid = batch_clip_uids[0]

one_clip_result = preprocess_one_clip_for_cache(
    test_clip_uid,
    force_recompute=False,
)

one_clip_result

{'clip_uid': 'dfe962ab-6aa7-4888-9378-796817a30ab6',
 'video_uid': 'e14e03f8-13e4-4df2-87b0-e1ad8a175f7c',
 'status': 'skipped_fully_cached',
 'error': None,
 'index_cache_exists_before': True,
 'query_cache_exists_before': True,
 'query_metadata_exists_before': True,
 'num_index_frames': None,
 'num_queries': 6}

In [16]:
# ---------------- 9. Batch-safe preprocessing loop ----------------

def run_preprocess_batch_for_cache(batch_clip_uids, batch_id, force_recompute=False):
    """
    Run preprocessing for one batch.
    Saves progress after each clip.
    """
    manifest_path = RESULTS_DIR / f"preprocess_manifest_nclips{N_CLIPS}_batch{batch_id}.csv"
    errors_path = RESULTS_DIR / f"preprocess_errors_nclips{N_CLIPS}_batch{batch_id}.json"

    rows = []
    errors = []

    for clip_uid in tqdm(batch_clip_uids, desc=f"Preprocess batch {batch_id}"):
        row = preprocess_one_clip_for_cache(
            clip_uid,
            force_recompute=force_recompute,
        )

        rows.append(row)

        if row["status"] == "error":
            errors.append(row)

        # Save progress after every clip
        pd.DataFrame(rows).to_csv(manifest_path, index=False)

        with open(errors_path, "w") as f:
            json.dump(errors, f, indent=2)

        print(
            f"Finished clip {clip_uid} | "
            f"status={row['status']} | "
            f"errors={len(errors)}"
        )

    manifest_df = pd.DataFrame(rows)

    print("\nFinished batch:", batch_id)
    print("Total clips:", len(manifest_df))
    print("Success:", (manifest_df["status"] == "success").sum())
    print("Skipped fully cached:", (manifest_df["status"] == "skipped_fully_cached").sum())
    print("Errors:", (manifest_df["status"] == "error").sum())
    print("Saved manifest to:", manifest_path)
    print("Saved errors to:", errors_path)

    return manifest_df

In [16]:
# ---------------- 10. Run full batch 0 ----------------

batch_manifest_df = run_preprocess_batch_for_cache(
    batch_clip_uids=batch_clip_uids,
    batch_id=BATCH_ID,
    force_recompute=False,
)

display(batch_manifest_df)
display(batch_manifest_df["status"].value_counts(dropna=False))

Preprocess batch 0:   0%|          | 0/60 [00:00<?, ?it/s]

Finished clip dfe962ab-6aa7-4888-9378-796817a30ab6 | status=skipped_fully_cached | errors=0
Finished clip b157270d-c67a-41db-91ec-32bb3ec738fc | status=skipped_fully_cached | errors=0
Finished clip 7bb41f34-1743-42ad-97d1-c54f4472b85e | status=skipped_fully_cached | errors=0
Finished clip ea365183-5841-4f00-9a2e-1b81c5e09c98 | status=skipped_fully_cached | errors=0
Finished clip b961eb69-cfdd-4d5c-b427-4b667fedfcda | status=skipped_fully_cached | errors=0
Finished clip c8070026-c3dc-43d3-8d36-65c5d9a1317f | status=skipped_fully_cached | errors=0
Finished clip bd87e5e6-d2e3-4e8e-8c0d-dc57a93f8b76 | status=skipped_fully_cached | errors=0
Finished clip 11acf51c-9a44-4c77-9e16-46925c2e66de | status=skipped_fully_cached | errors=0
Finished clip 5e961d49-5eb3-4d6d-b2f0-c9be437ba623 | status=skipped_fully_cached | errors=0
Finished clip 7a954faf-102f-4aba-9317-96d73c8fd104 | status=skipped_fully_cached | errors=0
Finished clip cbd49e40-6bbb-4083-b0e2-b6f37d92ad1c | status=skipped_fully_cached

,clip_uid,video_uid,status,error,index_cache_exists_before,query_cache_exists_before,query_metadata_exists_before,num_index_frames,num_queries
0,dfe962ab-6aa7-4888-9378-796817a30ab6,e14e03f8-13e4-4df2-87b0-e1ad8a175f7c,skipped_fully_cached,None,True,True,True,None,6
1,b157270d-c67a-41db-91ec-32bb3ec738fc,eb486644-f46f-44cf-8e4b-73245b2fc02e,skipped_fully_cached,None,True,True,True,None,6
2,7bb41f34-1743-42ad-97d1-c54f4472b85e,eb486644-f46f-44cf-8e4b-73245b2fc02e,skipped_fully_cached,None,True,True,True,None,3
3,ea365183-5841-4f00-9a2e-1b81c5e09c98,eb486644-f46f-44cf-8e4b-73245b2fc02e,skipped_fully_cached,None,True,True,True,None,3
4,b961eb69-cfdd-4d5c-b427-4b667fedfcda,d8c40e36-c9fe-4c79-aa4c-271ab6427c2d,skipped_fully_cached,None,True,True,True,None,3
5,c8070026-c3dc-43d3-8d36-65c5d9a1317f,8a6a3316-d682-4a76-81db-b244081765c9,skipped_fully_cached,None,True,True,True,None,3
6,bd87e5e6-d2e3-4e8e-8c0d-dc57a93f8b76,8a6a3316-d682-4a76-81db-b244081765c9,skipped_fully_cached,None,True,True,True,None,6
7,11acf51c-9a44-4c77-9e16-46925c2e66de,f20bca30-b7f0-4d2f-92ae-3982602f05df,skipped_fully_cached,None,True,True,True,None,3
8,5e961d49-5eb3-4d6d-b2f0-c9be437ba623,7e2a3795-28a5-4314-a4d7-b91807812ed1,skipped_fully_cached,None,True,True,True,None,3
9,7a954faf-102f-4aba-9317-96d73c8fd104,32991c95-5acb-4d1b-a142-98c2fe27cbfd,skipped_fully_cached,None,True,True,True,None,3


status
skipped_fully_cached    60
Name: count, dtype: int64

In [17]:
# ---------------- 11. Switch to batch 1 ----------------

BATCH_ID = 1

all_workplan_clips = list(workplan.keys())

start_idx = BATCH_ID * BATCH_SIZE_CLIPS
end_idx = min(start_idx + BATCH_SIZE_CLIPS, len(all_workplan_clips))

batch_clip_uids = all_workplan_clips[start_idx:end_idx]

print("Total workplan clips:", len(all_workplan_clips))
print("Batch ID:", BATCH_ID)
print("Batch start:", start_idx)
print("Batch end:", end_idx)
print("Batch clips:", len(batch_clip_uids))
print("First 5 batch clips:", batch_clip_uids[:5])

Total workplan clips: 300
Batch ID: 1
Batch start: 60
Batch end: 120
Batch clips: 60
First 5 batch clips: ['dbd760b1-f99a-4a74-9f67-50579e516532', '8d22e91b-ed98-4c87-8a25-c937236ca745', '81da71e1-0e80-43cc-8752-292d231e70a2', '67c47998-eabe-4b31-9096-b286c18e1beb', 'fd82660b-9650-4326-93fc-08b1b950ac9d']


In [18]:
# ---------------- 12. Check cache status for batch 1 ----------------

batch_cache_status_df = pd.DataFrame(
    [check_clip_cache_status(clip_uid) for clip_uid in batch_clip_uids]
)

display(batch_cache_status_df.head())

print("Batch clips:", len(batch_cache_status_df))
print("Index caches exist:", batch_cache_status_df["index_cache_exists"].sum())
print("Query caches exist:", batch_cache_status_df["query_cache_exists"].sum())
print("Query metadata exists:", batch_cache_status_df["query_metadata_exists"].sum())
print("Fully cached:", batch_cache_status_df["fully_cached"].sum())

,clip_uid,index_cache_exists,query_cache_exists,query_metadata_exists,fully_cached
0,dbd760b1-f99a-4a74-9f67-50579e516532,False,False,False,False
1,8d22e91b-ed98-4c87-8a25-c937236ca745,False,False,False,False
2,81da71e1-0e80-43cc-8752-292d231e70a2,False,False,False,False
3,67c47998-eabe-4b31-9096-b286c18e1beb,True,True,True,True
4,fd82660b-9650-4326-93fc-08b1b950ac9d,False,False,False,False


Batch clips: 60
Index caches exist: 4
Query caches exist: 4
Query metadata exists: 4
Fully cached: 4


In [21]:
# ---------------- 13. Run full batch 1 ----------------

batch_manifest_df = run_preprocess_batch_for_cache(
    batch_clip_uids=batch_clip_uids,
    batch_id=BATCH_ID,
    force_recompute=False,
)

display(batch_manifest_df)
display(batch_manifest_df["status"].value_counts(dropna=False))

Preprocess batch 0:   0%|          | 0/60 [00:00<?, ?it/s]

Finished clip dbd760b1-f99a-4a74-9f67-50579e516532 | status=skipped_fully_cached | errors=0
Finished clip 8d22e91b-ed98-4c87-8a25-c937236ca745 | status=skipped_fully_cached | errors=0
Finished clip 81da71e1-0e80-43cc-8752-292d231e70a2 | status=skipped_fully_cached | errors=0
Finished clip 67c47998-eabe-4b31-9096-b286c18e1beb | status=skipped_fully_cached | errors=0
Finished clip fd82660b-9650-4326-93fc-08b1b950ac9d | status=skipped_fully_cached | errors=0
Finished clip c77d3785-22f9-46c9-bf6d-9f7196356749 | status=skipped_fully_cached | errors=0
Finished clip 9f661edf-a6f6-444e-b168-e8428044b1d7 | status=skipped_fully_cached | errors=0
Finished clip f1eb7035-ce79-4be2-81bc-118eb1ad2887 | status=skipped_fully_cached | errors=0
Finished clip d9d45d26-743b-4e26-8137-416372bdaddb | status=skipped_fully_cached | errors=0
Finished clip 09d85b55-d473-49c1-9ebc-e871714785cd | status=skipped_fully_cached | errors=0
Finished clip 706c54f8-2740-40bb-a341-651b708392fa | status=skipped_fully_cached

,clip_uid,video_uid,status,error,index_cache_exists_before,query_cache_exists_before,query_metadata_exists_before,num_index_frames,num_queries
0,dbd760b1-f99a-4a74-9f67-50579e516532,d79f9434-2456-4a01-bf72-d597a5668e86,skipped_fully_cached,None,True,True,True,None,3
1,8d22e91b-ed98-4c87-8a25-c937236ca745,73773748-14ac-40ba-9ef8-d5a70865aeea,skipped_fully_cached,None,True,True,True,None,6
2,81da71e1-0e80-43cc-8752-292d231e70a2,b707b510-4ccb-4912-b662-5e2dbdaa6ef2,skipped_fully_cached,None,True,True,True,None,3
3,67c47998-eabe-4b31-9096-b286c18e1beb,abbc5555-7816-4288-8d4d-f42b0683fc44,skipped_fully_cached,None,True,True,True,None,6
4,fd82660b-9650-4326-93fc-08b1b950ac9d,0b245a61-32d6-4b14-897c-724adad5b231,skipped_fully_cached,None,True,True,True,None,3
5,c77d3785-22f9-46c9-bf6d-9f7196356749,eea4b231-055e-4c4e-8f95-8dd043601da6,skipped_fully_cached,None,True,True,True,None,3
6,9f661edf-a6f6-444e-b168-e8428044b1d7,316ad960-e087-4d33-ba02-08e83c20e199,skipped_fully_cached,None,True,True,True,None,6
7,f1eb7035-ce79-4be2-81bc-118eb1ad2887,d405ceed-68da-4e50-a0ef-5d2995a94e5d,skipped_fully_cached,None,True,True,True,None,3
8,d9d45d26-743b-4e26-8137-416372bdaddb,d405ceed-68da-4e50-a0ef-5d2995a94e5d,skipped_fully_cached,None,True,True,True,None,6
9,09d85b55-d473-49c1-9ebc-e871714785cd,d405ceed-68da-4e50-a0ef-5d2995a94e5d,skipped_fully_cached,None,True,True,True,None,6


status
skipped_fully_cached    60
Name: count, dtype: int64

In [22]:
# ---------------- 14. Switch to batch 2 ----------------

BATCH_ID = 2

all_workplan_clips = list(workplan.keys())

start_idx = BATCH_ID * BATCH_SIZE_CLIPS
end_idx = min(start_idx + BATCH_SIZE_CLIPS, len(all_workplan_clips))

batch_clip_uids = all_workplan_clips[start_idx:end_idx]

print("Total workplan clips:", len(all_workplan_clips))
print("Batch ID:", BATCH_ID)
print("Batch start:", start_idx)
print("Batch end:", end_idx)
print("Batch clips:", len(batch_clip_uids))
print("First 5 batch clips:", batch_clip_uids[:5])

Total workplan clips: 300
Batch ID: 2
Batch start: 120
Batch end: 180
Batch clips: 60
First 5 batch clips: ['617e8d7c-8764-4508-ba9b-c993282c70cd', 'b58bc49e-d3ac-4995-baa0-218a14e4c2fa', '81221073-c095-4129-b095-97f3d6e4a6b1', '5e66abe8-a72b-43f3-b6bf-fc05a45ff80f', '5ab23181-c60a-4478-9085-2a450a24b65b']


In [23]:
# ---------------- 15. Check cache status for batch 2 ----------------

batch_cache_status_df = pd.DataFrame(
    [check_clip_cache_status(clip_uid) for clip_uid in batch_clip_uids]
)

display(batch_cache_status_df.head())

print("Batch clips:", len(batch_cache_status_df))
print("Index caches exist:", batch_cache_status_df["index_cache_exists"].sum())
print("Query caches exist:", batch_cache_status_df["query_cache_exists"].sum())
print("Query metadata exists:", batch_cache_status_df["query_metadata_exists"].sum())
print("Fully cached:", batch_cache_status_df["fully_cached"].sum())

,clip_uid,index_cache_exists,query_cache_exists,query_metadata_exists,fully_cached
0,617e8d7c-8764-4508-ba9b-c993282c70cd,False,False,False,False
1,b58bc49e-d3ac-4995-baa0-218a14e4c2fa,False,False,False,False
2,81221073-c095-4129-b095-97f3d6e4a6b1,False,False,False,False
3,5e66abe8-a72b-43f3-b6bf-fc05a45ff80f,False,False,False,False
4,5ab23181-c60a-4478-9085-2a450a24b65b,False,False,False,False


Batch clips: 60
Index caches exist: 5
Query caches exist: 5
Query metadata exists: 5
Fully cached: 5


In [26]:
# ---------------- 16. Run full batch 2 ----------------

batch_manifest_df = run_preprocess_batch_for_cache(
    batch_clip_uids=batch_clip_uids,
    batch_id=BATCH_ID,
    force_recompute=False,
)

display(batch_manifest_df)
display(batch_manifest_df["status"].value_counts(dropna=False))

Preprocess batch 2:   0%|          | 0/60 [00:00<?, ?it/s]

Finished clip 617e8d7c-8764-4508-ba9b-c993282c70cd | status=skipped_fully_cached | errors=0
Finished clip b58bc49e-d3ac-4995-baa0-218a14e4c2fa | status=skipped_fully_cached | errors=0
Finished clip 81221073-c095-4129-b095-97f3d6e4a6b1 | status=skipped_fully_cached | errors=0
Finished clip 5e66abe8-a72b-43f3-b6bf-fc05a45ff80f | status=skipped_fully_cached | errors=0
Finished clip 5ab23181-c60a-4478-9085-2a450a24b65b | status=skipped_fully_cached | errors=0
Finished clip 99ddfcb3-bb2a-45d3-903e-d7e858969957 | status=skipped_fully_cached | errors=0
Finished clip 14ea5c3e-bbb5-44cf-abcc-e70e1223ad74 | status=skipped_fully_cached | errors=0
Finished clip 5771acd7-c7a9-4350-8064-7763bf111149 | status=skipped_fully_cached | errors=0
Finished clip aef0725a-1005-4a17-ad33-4522808c3a17 | status=skipped_fully_cached | errors=0
Finished clip f835883d-2ef1-427d-9c2f-5d9e7670cf5e | status=skipped_fully_cached | errors=0
Finished clip 629cb4e4-65a5-45f5-909d-2ff93e57a0bf | status=skipped_fully_cached

,clip_uid,video_uid,status,error,index_cache_exists_before,query_cache_exists_before,query_metadata_exists_before,num_index_frames,num_queries
0,617e8d7c-8764-4508-ba9b-c993282c70cd,25beabfc-48c0-4f81-b419-c47297aecf6f,skipped_fully_cached,None,True,True,True,None,3
1,b58bc49e-d3ac-4995-baa0-218a14e4c2fa,25beabfc-48c0-4f81-b419-c47297aecf6f,skipped_fully_cached,None,True,True,True,None,3
2,81221073-c095-4129-b095-97f3d6e4a6b1,412763ce-9a81-4bb7-b713-80fe2713e1a8,skipped_fully_cached,None,True,True,True,None,3
3,5e66abe8-a72b-43f3-b6bf-fc05a45ff80f,0aec2b0c-18e1-453b-a6ed-e484698d41df,skipped_fully_cached,None,True,True,True,None,3
4,5ab23181-c60a-4478-9085-2a450a24b65b,65ffa888-a7c8-43a7-afd8-13b2df7f7f04,skipped_fully_cached,None,True,True,True,None,3
5,99ddfcb3-bb2a-45d3-903e-d7e858969957,f88c7f52-dfb9-4a17-ac15-f3ec05de4f85,skipped_fully_cached,None,True,True,True,None,3
6,14ea5c3e-bbb5-44cf-abcc-e70e1223ad74,f88c7f52-dfb9-4a17-ac15-f3ec05de4f85,skipped_fully_cached,None,True,True,True,None,3
7,5771acd7-c7a9-4350-8064-7763bf111149,975427a5-9d66-4bf6-aa94-1266297cc5e6,skipped_fully_cached,None,True,True,True,None,3
8,aef0725a-1005-4a17-ad33-4522808c3a17,b1543ea3-e86b-4545-9195-f1bf83e02b41,skipped_fully_cached,None,True,True,True,None,3
9,f835883d-2ef1-427d-9c2f-5d9e7670cf5e,5cab806a-3794-4578-94f2-33c3093151d1,skipped_fully_cached,None,True,True,True,None,3


status
skipped_fully_cached    60
Name: count, dtype: int64

In [27]:
# ---------------- 17. Switch to batch 3 ----------------

BATCH_ID = 3

all_workplan_clips = list(workplan.keys())

start_idx = BATCH_ID * BATCH_SIZE_CLIPS
end_idx = min(start_idx + BATCH_SIZE_CLIPS, len(all_workplan_clips))

batch_clip_uids = all_workplan_clips[start_idx:end_idx]

print("Total workplan clips:", len(all_workplan_clips))
print("Batch ID:", BATCH_ID)
print("Batch start:", start_idx)
print("Batch end:", end_idx)
print("Batch clips:", len(batch_clip_uids))
print("First 5 batch clips:", batch_clip_uids[:5])

Total workplan clips: 300
Batch ID: 3
Batch start: 180
Batch end: 240
Batch clips: 60
First 5 batch clips: ['cafcebbb-d1c1-439b-8228-3bd95b8d3b7f', '16fce853-9528-4c39-9f88-b0c61f30d01d', '9b9b7994-8a7a-43ac-9382-744664ee3e66', 'fe71b934-322a-45a1-862c-cd7c2db2c354', '8f1770e5-612c-4047-a7b7-a6157460bc24']


In [28]:
# ---------------- 18. Check cache status for batch 3 ----------------

batch_cache_status_df = pd.DataFrame(
    [check_clip_cache_status(clip_uid) for clip_uid in batch_clip_uids]
)

display(batch_cache_status_df.head())

print("Batch clips:", len(batch_cache_status_df))
print("Index caches exist:", batch_cache_status_df["index_cache_exists"].sum())
print("Query caches exist:", batch_cache_status_df["query_cache_exists"].sum())
print("Query metadata exists:", batch_cache_status_df["query_metadata_exists"].sum())
print("Fully cached:", batch_cache_status_df["fully_cached"].sum())

,clip_uid,index_cache_exists,query_cache_exists,query_metadata_exists,fully_cached
0,cafcebbb-d1c1-439b-8228-3bd95b8d3b7f,False,False,False,False
1,16fce853-9528-4c39-9f88-b0c61f30d01d,False,False,False,False
2,9b9b7994-8a7a-43ac-9382-744664ee3e66,False,False,False,False
3,fe71b934-322a-45a1-862c-cd7c2db2c354,False,False,False,False
4,8f1770e5-612c-4047-a7b7-a6157460bc24,False,False,False,False


Batch clips: 60
Index caches exist: 2
Query caches exist: 2
Query metadata exists: 2
Fully cached: 2


In [30]:
# ---------------- 19. Run full batch 3 ----------------

batch_manifest_df = run_preprocess_batch_for_cache(
    batch_clip_uids=batch_clip_uids,
    batch_id=BATCH_ID,
    force_recompute=False,
)

display(batch_manifest_df)
display(batch_manifest_df["status"].value_counts(dropna=False))

Preprocess batch 3:   0%|          | 0/60 [00:00<?, ?it/s]

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.
Finished clip cafcebbb-d1c1-439b-8228-3bd95b8d3b7f | status=skipped_fully_cached | errors=0
Finished clip 16fce853-9528-4c39-9f88-b0c61f30d01d | status=skipped_fully_cached | errors=0
Finished clip 9b9b7994-8a7a-43ac-9382-744664ee3e66 | status=skipped_fully_cached | errors=0
Finished clip fe71b934-322a-45a1-862c-cd7c2db2c354 | status=skipped_fully_cached | errors=0
Finished clip 8f1770e5-612c-4047-a7b7-a6157460bc24 | status=skipped_fully_cached | errors=0
Finished clip f9cbfb3a-d2e2-4f30-9c01-ee341b5e8887 | status=skipped_fully_cached | errors=0
Finished clip 63a1c028-3919-462b-9c97-9a22c5296eae | status=skipped_fully_cached | errors=0
Finished clip 6bfb9000-73c2-4698-8757-889251a4b5e9 | status=skipped_fully_cached | errors=0
Finished clip 02089b3c-2c7a-430b-9c7c-9c3f39249ec7 | status=skipped_fully_cached | errors=0
Finished clip f54175da-19e5-4e

,clip_uid,video_uid,status,error,index_cache_exists_before,query_cache_exists_before,query_metadata_exists_before,num_index_frames,num_queries
0,cafcebbb-d1c1-439b-8228-3bd95b8d3b7f,2bc60910-3a1f-4498-a579-a1bcf105aee7,skipped_fully_cached,NaN,True,True,True,NaN,3.0
1,16fce853-9528-4c39-9f88-b0c61f30d01d,2bc60910-3a1f-4498-a579-a1bcf105aee7,skipped_fully_cached,NaN,True,True,True,NaN,3.0
2,9b9b7994-8a7a-43ac-9382-744664ee3e66,1938c632-f575-49dd-8ae0-e48dbb467920,skipped_fully_cached,NaN,True,True,True,NaN,3.0
3,fe71b934-322a-45a1-862c-cd7c2db2c354,45ef4272-053f-46f2-9343-1709be6d188c,skipped_fully_cached,NaN,True,True,True,NaN,3.0
4,8f1770e5-612c-4047-a7b7-a6157460bc24,45ef4272-053f-46f2-9343-1709be6d188c,skipped_fully_cached,NaN,True,True,True,NaN,3.0
5,f9cbfb3a-d2e2-4f30-9c01-ee341b5e8887,2763dbfc-8488-4264-b31f-4b12096c4c4b,skipped_fully_cached,NaN,True,True,True,NaN,6.0
6,63a1c028-3919-462b-9c97-9a22c5296eae,c1b75b09-d3e8-4de4-996f-6c49d0efcd10,skipped_fully_cached,NaN,True,True,True,NaN,3.0
7,6bfb9000-73c2-4698-8757-889251a4b5e9,8672b38a-7682-4244-ad7a-a336c81c2d0e,skipped_fully_cached,NaN,True,True,True,NaN,3.0
8,02089b3c-2c7a-430b-9c7c-9c3f39249ec7,18a3840b-7463-43c4-9aa9-b1d8e486fa84,skipped_fully_cached,NaN,True,True,True,NaN,3.0
9,f54175da-19e5-4ee7-97b3-75baa247b713,115774b6-534d-444f-b7aa-d1b834eb0ee7,skipped_fully_cached,NaN,True,True,True,NaN,6.0


status
skipped_fully_cached    59
error                    1
Name: count, dtype: int64

In [31]:
# ---------------- 20. Switch to batch 4 ----------------

BATCH_ID = 4

all_workplan_clips = list(workplan.keys())

start_idx = BATCH_ID * BATCH_SIZE_CLIPS
end_idx = min(start_idx + BATCH_SIZE_CLIPS, len(all_workplan_clips))

batch_clip_uids = all_workplan_clips[start_idx:end_idx]

print("Total workplan clips:", len(all_workplan_clips))
print("Batch ID:", BATCH_ID)
print("Batch start:", start_idx)
print("Batch end:", end_idx)
print("Batch clips:", len(batch_clip_uids))
print("First 5 batch clips:", batch_clip_uids[:5])

Total workplan clips: 300
Batch ID: 4
Batch start: 240
Batch end: 300
Batch clips: 60
First 5 batch clips: ['cc994557-a65b-48a9-94c6-122ede470347', '4a491385-b165-49ec-a23b-7b0ed26737f9', 'dc3ba39f-62f4-480b-a187-ca723c8666bd', 'bc01d0dd-be04-474e-9fc6-de83292a5062', 'bcd1ea04-bde8-4769-acb4-3d504183cbf2']


In [32]:
# ---------------- 21. Check cache status for batch 4 ----------------

batch_cache_status_df = pd.DataFrame(
    [check_clip_cache_status(clip_uid) for clip_uid in batch_clip_uids]
)

display(batch_cache_status_df.head())

print("Batch clips:", len(batch_cache_status_df))
print("Index caches exist:", batch_cache_status_df["index_cache_exists"].sum())
print("Query caches exist:", batch_cache_status_df["query_cache_exists"].sum())
print("Query metadata exists:", batch_cache_status_df["query_metadata_exists"].sum())
print("Fully cached:", batch_cache_status_df["fully_cached"].sum())

,clip_uid,index_cache_exists,query_cache_exists,query_metadata_exists,fully_cached
0,cc994557-a65b-48a9-94c6-122ede470347,False,False,False,False
1,4a491385-b165-49ec-a23b-7b0ed26737f9,False,False,False,False
2,dc3ba39f-62f4-480b-a187-ca723c8666bd,False,False,False,False
3,bc01d0dd-be04-474e-9fc6-de83292a5062,False,False,False,False
4,bcd1ea04-bde8-4769-acb4-3d504183cbf2,False,False,False,False


Batch clips: 60
Index caches exist: 3
Query caches exist: 3
Query metadata exists: 3
Fully cached: 3


In [36]:
# ---------------- 22. Run final batch 4 ----------------

batch_manifest_df = run_preprocess_batch_for_cache(
    batch_clip_uids=batch_clip_uids,
    batch_id=BATCH_ID,
    force_recompute=False,
)

display(batch_manifest_df)
display(batch_manifest_df["status"].value_counts(dropna=False))

Preprocess batch 4:   0%|          | 0/60 [00:00<?, ?it/s]

Finished clip cc994557-a65b-48a9-94c6-122ede470347 | status=skipped_fully_cached | errors=0
Finished clip 4a491385-b165-49ec-a23b-7b0ed26737f9 | status=skipped_fully_cached | errors=0
Finished clip dc3ba39f-62f4-480b-a187-ca723c8666bd | status=skipped_fully_cached | errors=0
Finished clip bc01d0dd-be04-474e-9fc6-de83292a5062 | status=skipped_fully_cached | errors=0
Finished clip bcd1ea04-bde8-4769-acb4-3d504183cbf2 | status=skipped_fully_cached | errors=0
Finished clip cf82756f-8e87-443a-972c-405b25858e78 | status=skipped_fully_cached | errors=0
Finished clip 2ddf340f-de44-4f5d-9994-1e0f7db05caf | status=skipped_fully_cached | errors=0
Finished clip 7380c46e-cb33-4d49-bf73-a7bfe57c3feb | status=skipped_fully_cached | errors=0
Finished clip f14ea4c5-2a1a-4855-a212-3b2abe7aa0f2 | status=skipped_fully_cached | errors=0
Finished clip 4e9b50a0-1a9b-4e23-9f3b-4bfde1a7cae9 | status=skipped_fully_cached | errors=0
Finished clip 17043256-42c5-4e4f-846b-7923afe3ead4 | status=skipped_fully_cached

,clip_uid,video_uid,status,error,index_cache_exists_before,query_cache_exists_before,query_metadata_exists_before,num_index_frames,num_queries
0,cc994557-a65b-48a9-94c6-122ede470347,6e0c294c-d634-4804-8d41-cd6e9d79f0b9,skipped_fully_cached,None,True,True,True,None,3
1,4a491385-b165-49ec-a23b-7b0ed26737f9,3ee7070a-fe81-49a9-97a2-902268af9985,skipped_fully_cached,None,True,True,True,None,6
2,dc3ba39f-62f4-480b-a187-ca723c8666bd,ecc633d2-72b5-4bd0-a6a1-f1cedb21d757,skipped_fully_cached,None,True,True,True,None,3
3,bc01d0dd-be04-474e-9fc6-de83292a5062,f4f76ace-2b4b-4e7a-b380-cc08f7642223,skipped_fully_cached,None,True,True,True,None,3
4,bcd1ea04-bde8-4769-acb4-3d504183cbf2,2876b375-e848-412c-8a6f-0664cbab6a33,skipped_fully_cached,None,True,True,True,None,3
5,cf82756f-8e87-443a-972c-405b25858e78,8fec5995-c65d-4cae-adaf-45eb76956085,skipped_fully_cached,None,True,True,True,None,6
6,2ddf340f-de44-4f5d-9994-1e0f7db05caf,033c136a-7a79-4c7e-a0c3-15e4ad0a22cf,skipped_fully_cached,None,True,True,True,None,3
7,7380c46e-cb33-4d49-bf73-a7bfe57c3feb,e191e0de-e570-4925-9cbb-e05fe1132a47,skipped_fully_cached,None,True,True,True,None,3
8,f14ea4c5-2a1a-4855-a212-3b2abe7aa0f2,7b3bea48-bd62-46b7-888d-2cc6e4bbd11b,skipped_fully_cached,None,True,True,True,None,6
9,4e9b50a0-1a9b-4e23-9f3b-4bfde1a7cae9,5cae3bb3-1e1e-4b9f-82e8-87dbaad8f73d,skipped_fully_cached,None,True,True,True,None,6


status
skipped_fully_cached    60
Name: count, dtype: int64

In [37]:
# ---------------- 23. Final manifest summary for all 5 batches ----------------

manifest_paths = [
    RESULTS_DIR / f"preprocess_manifest_nclips{N_CLIPS}_batch{i}.csv"
    for i in range(5)
]

all_manifest_dfs = []

for i, path in enumerate(manifest_paths):
    print(f"Batch {i} manifest exists:", path.exists(), "|", path)

    if path.exists():
        df_i = pd.read_csv(path)
        df_i["batch_id"] = i
        all_manifest_dfs.append(df_i)

all_manifest_df = pd.concat(all_manifest_dfs, ignore_index=True)

print("\n================ Final preprocessing summary ================")
print("Total manifest rows:", len(all_manifest_df))
print("Unique clips:", all_manifest_df["clip_uid"].nunique())

print("\nStatus counts:")
display(all_manifest_df["status"].value_counts(dropna=False))

print("\nBatch x status:")
display(
    all_manifest_df
    .groupby(["batch_id", "status"])
    .size()
    .reset_index(name="count")
)

final_manifest_path = RESULTS_DIR / f"preprocess_manifest_nclips{N_CLIPS}_all_batches.csv"
all_manifest_df.to_csv(final_manifest_path, index=False)

print("\nSaved combined manifest to:", final_manifest_path)

Batch 0 manifest exists: True | /home/jupyter/results/retrieval_subset/preprocess_manifest_nclips300_batch0.csv
Batch 1 manifest exists: True | /home/jupyter/results/retrieval_subset/preprocess_manifest_nclips300_batch1.csv
Batch 2 manifest exists: True | /home/jupyter/results/retrieval_subset/preprocess_manifest_nclips300_batch2.csv
Batch 3 manifest exists: True | /home/jupyter/results/retrieval_subset/preprocess_manifest_nclips300_batch3.csv
Batch 4 manifest exists: True | /home/jupyter/results/retrieval_subset/preprocess_manifest_nclips300_batch4.csv

================ Final preprocessing summary ================
Total manifest rows: 300
Unique clips: 240

Status counts:


status
skipped_fully_cached    243
success                  56
error                     1
Name: count, dtype: int64


Batch x status:


,batch_id,status,count
0,0,skipped_fully_cached,60
1,1,skipped_fully_cached,4
2,1,success,56
3,2,skipped_fully_cached,60
4,3,error,1
5,3,skipped_fully_cached,59
6,4,skipped_fully_cached,60



Saved combined manifest to: /home/jupyter/results/retrieval_subset/preprocess_manifest_nclips300_all_batches.csv


In [38]:
# ---------------- 24. Final cache completeness check ----------------

all_300_clip_uids = list(workplan.keys())

final_cache_status_df = pd.DataFrame(
    [check_clip_cache_status(clip_uid) for clip_uid in all_300_clip_uids]
)

display(final_cache_status_df.head())

print("Total clips checked:", len(final_cache_status_df))
print("Index caches exist:", final_cache_status_df["index_cache_exists"].sum())
print("Query caches exist:", final_cache_status_df["query_cache_exists"].sum())
print("Query metadata exists:", final_cache_status_df["query_metadata_exists"].sum())
print("Fully cached:", final_cache_status_df["fully_cached"].sum())

missing_cache_df = final_cache_status_df[~final_cache_status_df["fully_cached"]].copy()

print("\nMissing / incomplete cache clips:", len(missing_cache_df))
display(missing_cache_df)

final_cache_status_path = RESULTS_DIR / f"cache_status_nclips{N_CLIPS}_final.csv"
final_cache_status_df.to_csv(final_cache_status_path, index=False)

print("\nSaved final cache status to:", final_cache_status_path)

,clip_uid,index_cache_exists,query_cache_exists,query_metadata_exists,fully_cached
0,dfe962ab-6aa7-4888-9378-796817a30ab6,True,True,True,True
1,b157270d-c67a-41db-91ec-32bb3ec738fc,True,True,True,True
2,7bb41f34-1743-42ad-97d1-c54f4472b85e,True,True,True,True
3,ea365183-5841-4f00-9a2e-1b81c5e09c98,True,True,True,True
4,b961eb69-cfdd-4d5c-b427-4b667fedfcda,True,True,True,True


Total clips checked: 300
Index caches exist: 299
Query caches exist: 299
Query metadata exists: 299
Fully cached: 299

Missing / incomplete cache clips: 1


,clip_uid,index_cache_exists,query_cache_exists,query_metadata_exists,fully_cached
239,18dad6e7-5969-4573-a1b3-f4ccfc53c350,False,False,False,False



Saved final cache status to: /home/jupyter/results/retrieval_subset/cache_status_nclips300_final.csv


In [17]:
print("workplan exists:", "workplan" in globals())
print("check_clip_cache_status exists:", "check_clip_cache_status" in globals())
print("preprocess_one_clip_for_cache exists:", "preprocess_one_clip_for_cache" in globals())
print("N_CLIPS:", N_CLIPS)
print("Workplan clips:", len(workplan))

workplan exists: True
check_clip_cache_status exists: True
preprocess_one_clip_for_cache exists: True
N_CLIPS: 300
Workplan clips: 300


In [18]:
# ---------------- Repair the one missing cache clip ----------------

missing_clip_uid = "18dad6e7-5969-4573-a1b3-f4ccfc53c350"

print("Before repair:")
before_status = check_clip_cache_status(missing_clip_uid)
print(before_status)

repair_result = preprocess_one_clip_for_cache(
    missing_clip_uid,
    force_recompute=False,
)

print("\nRepair result:")
print(repair_result)

print("\nAfter repair:")
after_status = check_clip_cache_status(missing_clip_uid)
print(after_status)

Before repair:
{'clip_uid': '18dad6e7-5969-4573-a1b3-f4ccfc53c350', 'index_cache_exists': False, 'query_cache_exists': False, 'query_metadata_exists': False, 'fully_cached': False}
Building index embedding cache for clip: 18dad6e7-5969-4573-a1b3-f4ccfc53c350
Video already exists locally: /home/jupyter/tmp/retrieval_work/6e0c294c-d634-4804-8d41-cd6e9d79f0b9.mp4


Extracting frames for 18dad6e7-5969-4573-a1b3-f4ccfc53c350:   0%|          | 0/300 [00:00<?, ?it/s]

Number of index frames: 300


Embedding images:   0%|          | 0/5 [00:00<?, ?it/s]

Saved index embeddings to: /home/jupyter/results/retrieval_subset/cache/index_frame_embeddings/18dad6e7-5969-4573-a1b3-f4ccfc53c350_index_embeddings.npz
CLIP index embedding shape: (300, 512)
BLIP index embedding shape: (300, 768)
Building query embedding cache for clip: 18dad6e7-5969-4573-a1b3-f4ccfc53c350
Number of visual crop queries: 3


Embedding images:   0%|          | 0/1 [00:00<?, ?it/s]

Saved query embeddings to: /home/jupyter/results/retrieval_subset/cache/query_embeddings/18dad6e7-5969-4573-a1b3-f4ccfc53c350_query_embeddings.npz
Saved query metadata to: /home/jupyter/results/retrieval_subset/cache/query_embeddings/18dad6e7-5969-4573-a1b3-f4ccfc53c350_query_metadata.parquet
CLIP query embedding shape: (3, 512)
BLIP query embedding shape: (3, 768)

Repair result:
{'clip_uid': '18dad6e7-5969-4573-a1b3-f4ccfc53c350', 'video_uid': '6e0c294c-d634-4804-8d41-cd6e9d79f0b9', 'status': 'success', 'error': None, 'index_cache_exists_before': False, 'query_cache_exists_before': False, 'query_metadata_exists_before': False, 'num_index_frames': 300, 'num_queries': 3, 'index_cache_exists_after': True, 'query_cache_exists_after': True, 'query_metadata_exists_after': True, 'fully_cached_after': True}

After repair:
{'clip_uid': '18dad6e7-5969-4573-a1b3-f4ccfc53c350', 'index_cache_exists': True, 'query_cache_exists': True, 'query_metadata_exists': True, 'fully_cached': True}


In [19]:
# ---------------- Re-check final cache completeness ----------------

all_300_clip_uids = list(workplan.keys())

final_cache_status_df = pd.DataFrame(
    [check_clip_cache_status(clip_uid) for clip_uid in all_300_clip_uids]
)

display(final_cache_status_df.head())

print("Total clips checked:", len(final_cache_status_df))
print("Unique clips checked:", final_cache_status_df["clip_uid"].nunique())
print("Index caches exist:", final_cache_status_df["index_cache_exists"].sum())
print("Query caches exist:", final_cache_status_df["query_cache_exists"].sum())
print("Query metadata exists:", final_cache_status_df["query_metadata_exists"].sum())
print("Fully cached:", final_cache_status_df["fully_cached"].sum())

missing_cache_df = final_cache_status_df[~final_cache_status_df["fully_cached"]].copy()

print("\nMissing / incomplete cache clips:", len(missing_cache_df))
display(missing_cache_df)

final_cache_status_path = RESULTS_DIR / f"cache_status_nclips{N_CLIPS}_final_after_repair.csv"
final_cache_status_df.to_csv(final_cache_status_path, index=False)

print("\nSaved final cache status to:", final_cache_status_path)

,clip_uid,index_cache_exists,query_cache_exists,query_metadata_exists,fully_cached
0,dfe962ab-6aa7-4888-9378-796817a30ab6,True,True,True,True
1,b157270d-c67a-41db-91ec-32bb3ec738fc,True,True,True,True
2,7bb41f34-1743-42ad-97d1-c54f4472b85e,True,True,True,True
3,ea365183-5841-4f00-9a2e-1b81c5e09c98,True,True,True,True
4,b961eb69-cfdd-4d5c-b427-4b667fedfcda,True,True,True,True


Total clips checked: 300
Unique clips checked: 300
Index caches exist: 300
Query caches exist: 300
Query metadata exists: 300
Fully cached: 300

Missing / incomplete cache clips: 0


,clip_uid,index_cache_exists,query_cache_exists,query_metadata_exists,fully_cached



Saved final cache status to: /home/jupyter/results/retrieval_subset/cache_status_nclips300_final_after_repair.csv
